# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\ASUS\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\ASUS\data-analysis-course\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied_mask = home_outcomes == 3
home_denied = home_denied_mask.sum()
home_denial_rate = home_denied / home_n

home_approved_mask = home_outcomes.isin([1, 2])
home_approved = home_approved_mask.sum()

assert home_denied + home_approved == home_n, "Сумма отказов и одобрений не равна общему числу!"
assert 0 <= home_denial_rate <= 1, "Доля отказов вышла за пределы [0, 1]!"

print(f"Всего записей (home_n): {home_n}")
print(f"Отказов (код 3): {home_denied}")
print(f"Доля отказов: {home_denial_rate:.2%}")
print(f"Одобрений (коды 1 и 2): {home_approved}")

Всего записей (home_n): 6830
Отказов (код 3): 691
Доля отказов: 10.12%
Одобрений (коды 1 и 2): 6139


## H2. Одобрение без принятия предложения

In [4]:
home_not_accepted_mask = home_outcomes == 2
home_not_accepted = home_not_accepted_mask.sum()
home_not_accepted_rate = home_not_accepted / home_n

home_issued_mask = home_outcomes == 1
home_issued = home_issued_mask.sum()

assert home_approved - home_issued == home_not_accepted

home_approval_rate = home_approved / home_n
diff_pp = (home_approval_rate - (home_issued / home_n)) * 100

print(f"Код 2 (одобрено, не принято): {home_not_accepted}")
print(f"Доля кода 2: {home_not_accepted_rate:.2%}")
print(f"Выдач (код 1): {home_issued}")
print(f"Разность долей одобрений и выдач: {diff_pp:.2f} п.п.")

Код 2 (одобрено, не принято): 183
Доля кода 2: 2.68%
Выдач (код 1): 5956
Разность долей одобрений и выдач: 2.68 п.п.


## H3. Объяснение инструментов

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

## Текстовые коды для H+2

In [6]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом